In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
import torch
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
Q3_data = os.path.join(path, 'Q3_data.csv')
Q3_data = pd.read_csv(Q3_data)

print(f"Dataset shape: {Q3_data.shape}")

In [ ]:
# Task 2: Write your code here:
Q3_data.head()

In [ ]:
# Task 3: Write your code here:
Q3_data.info()

In [ ]:
# Task 4: Write your code here:
Q3_data.describe()

In [ ]:
# Task 1: Write your code here:
cols = ['P_2','D_39','B_1','B_2','R_1','S_3','D_41','B_3','D_42','D_43','D_137','D_138','D_139','D_140','D_141','D_142','D_143','D_144','D_145','Target']
df_clean = Q3_data[cols].copy()

for col in ['P_2','D_39','B_1','B_2','R_1','S_3','D_41','B_3','D_42','D_43','D_137','D_138','D_139','D_140','D_141','D_142','D_143','D_144','D_145','Target']:
    df_clean[col] = df_clean[col].fillna('unknown')

print("Missing values remaining:", df_clean.isnull().sum().sum())


In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(Q3_data)

In [ ]:
# Task 3: Write your code here:
categorical_cols = ['P_2','D_39','B_1','B_2','R_1','S_3','D_41','B_3','D_42','D_43','D_137','D_138','D_139','D_140','D_141','D_142','D_143','D_144','D_145','Target']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()

In [ ]:
# Task 4: Write your code here:
label_encoder = LabelEncoder()

for col in Q3_data.select_dtypes(include=["object"]).columns:
    Q3_data[col] = label_encoder.fit_transform(Q3_data[col])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(Q3_data)

In [ ]:
# Task 5: Write your code here:

In [ ]:
# Task 1: Write your code here:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
import torch

X = Q3_data.drop("D_140",axis=1)
y = Q3_data['D_140']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Task 2,3,4,5: Write your code here:
data = load_breast_cancer()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test  = torch.tensor(y_test, dtype=torch.float32)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)


first_sample, _ = train_dataset[0]
print(f"Shape of one sample: {first_sample.shape}")


In [ ]:
# Task 1: Write your code here:
weights = model.fc1.weight.data
feature_importance = torch.sum(torch.abs(weights), dim=0).cpu().numpy()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(12, 6))
plt.bar(range(len(feature_importance)), feature_importance)
plt.xlabel('Feature Index')
plt.ylabel('Importance')
plt.show()

golden_feature_index = np.argmax(feature_importance)
print(f"Golden Feature Index: {golden_feature_index}")
print(f"Max Importance Value: {feature_importance[golden_feature_index]}")

In [ ]:
# Task Bonus: Write your code here:
from catboost import CatBoostClassifier
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
import numpy as np

X_golden = X_train[:, [golden_feature_index]]
y_data = y_train

kf = KFold(n_splits=5, shuffle=True, random_state=42)
accuracies = []

print("Training CatBoost with Golden Feature only...")

for train_index, val_index in kf.split(X_golden):
    X_t, X_v = X_golden[train_index], X_golden[val_index]
    y_t, y_v = y_data[train_index], y_data[val_index]

    model = CatBoostClassifier(verbose=0)  # verbose=0 to silence training output
    model.fit(X_t, y_t)

    preds = model.predict(X_v)
    acc = accuracy_score(y_v, preds)
    accuracies.append(acc)

print(f"Average Accuracy with Golden Feature: {np.mean(accuracies):.4f}")